# 4.11 · 决策树回归 / Decision Tree Regressor

> **课程定位 / Where this fits**
> **Part 4 第 11 课, 树家族的起点**。KNN(4.10) 是"局部平均", 树是"**递归切分空间 + 每块取平均**"。树天生**抗维度诅咒、不需缩放、抓交互、可解释**——这些优点直接传给随机森林(4.12)和 GBDT(4.13), **表格数据的统治者**。
> Trees recursively partition space and average within each region. They resist the curse, need no scaling, capture interactions, and are interpretable — the foundation of forests and GBDT, rulers of tabular data.

> 💡 **面试相关 / Interview-relevant**
> - "决策树怎么选切分点 (回归)" ★★★★（最小化 MSE / 方差）
> - "树为什么不需要特征缩放" ★★★★★
> - "树怎么防过拟合 (剪枝/超参)" ★★★★
> - "树的优缺点" ★★★★

---

## 学习目标 / Learning Objectives
1. 理解 CART **回归树**的贪心切分准则（最小化加权 MSE）。
2. 从零实现一个简单回归树, 理解递归切分。
3. 看树为何产生**阶梯状**预测, 天生抓**交互**。
4. 用**深度/叶子大小/剪枝**控制过拟合。
5. 理解树的优缺点 → 为何要集成(4.12/4.13)。

## 目录 / TOC
1. [树 = 递归切分 + 区域平均 ⭐](#1)
2. [切分准则: 最小化加权 MSE ⭐](#2)
3. [💎 数据 + 从零实现](#3)
4. [对照 sklearn + 可视化切分](#4)
5. [深度 = 偏差-方差; 剪枝 ⭐](#5)
6. [特征重要性 + 不需缩放](#6)
7. [树的优缺点 → 为何集成](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 树 = 递归切分 + 区域平均 ⭐ / Recursive Partitioning

**决策树回归的思想**：
1. 找一个 **(特征, 阈值)** 把数据切成两块（如 carat < 1.0）
2. 对每块**递归**继续切
3. 叶子节点 = 一块区域, **预测值 = 该区域训练样本 y 的平均**

```
                carat < 1.0 ?
              /              \
        carat < 0.5 ?      carat < 1.5 ?
        /        \          /         \
    avg=800   avg=2500   avg=6000   avg=12000   (叶子=区域均值)
```

**和 KNN 对比**: KNN 用"距离"定义邻域, 树用"轴对齐切分"定义区域。树的区域是**矩形块**, 预测是**分段常数**（阶梯）。

**关键优点的来源**:
- **不需缩放**: 切分只看"特征值 > 阈值?"——单调变换不改变切分顺序(3.4 节铁律)
- **抓交互**: 嵌套切分天然表达"carat 大 **且** clarity 好 → 贵"
- **抗维度诅咒**: 每次只看一个特征, 不算全维距离
- **可解释**: 一条从根到叶的路径 = 一串 if-else 规则
Trees split on "feature > threshold", so monotonic transforms don't change splits (no scaling needed), nested splits capture interactions natively, and only one feature matters per split (curse-resistant).


<a id="2"></a>
## 2. 切分准则: 最小化加权 MSE ⭐ / Splitting Criterion

回归树怎么选最优 (特征, 阈值)？**贪心**地试所有可能切分, 选让**切分后加权 MSE 最小**的那个：

$$\text{MSE}_{\text{split}} = \frac{n_L}{n}\text{MSE}(L) + \frac{n_R}{n}\text{MSE}(R)$$

其中每块的 MSE = 该块 y 的方差（用块内均值预测的误差）。

**等价说法**: 选切分使**方差下降最多**（信息论里分类树用基尼/熵 1.6/0.11, 回归树用方差）。这是 CART (Classification And Regression Trees) 算法。
Equivalently: pick the split that reduces variance the most (classification trees use Gini/entropy; regression trees use variance).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(5000, random_state=0).reset_index(drop=True)
X = df[["carat","depth","table","x","y","z"]].values
y = df["price"].values
feat_names = ["carat","depth","table","x","y","z"]
print(f"Diamonds: {X.shape}")


<a id="3"></a>
## 3. 从零实现 / From Scratch


In [ ]:
# 从零回归树 (1D, 演示切分准则) / from-scratch regression tree (1D)
def best_split(x, y):
    best_mse, best_thresh = np.inf, None
    n = len(y)
    for t in np.unique(x)[1:]:               # 试每个候选阈值
        L, R = y[x < t], y[x >= t]
        if len(L) == 0 or len(R) == 0: continue
        mse = (len(L)*L.var() + len(R)*R.var()) / n   # 加权 MSE
        if mse < best_mse:
            best_mse, best_thresh = mse, t
    return best_thresh, best_mse

class SimpleTree:
    def __init__(self, max_depth=3, min_samples=10):
        self.max_depth, self.min_samples = max_depth, min_samples
    def fit(self, x, y, depth=0):
        self.value = y.mean()                 # 叶子默认=均值
        if depth >= self.max_depth or len(y) < self.min_samples:
            self.leaf = True; return self
        t, _ = best_split(x, y)
        if t is None: self.leaf = True; return self
        self.leaf, self.thresh = False, t
        self.left = SimpleTree(self.max_depth, self.min_samples).fit(x[x<t], y[x<t], depth+1)
        self.right = SimpleTree(self.max_depth, self.min_samples).fit(x[x>=t], y[x>=t], depth+1)
        return self
    def predict_one(self, xi):
        if self.leaf: return self.value
        return (self.left if xi < self.thresh else self.right).predict_one(xi)
    def predict(self, x): return np.array([self.predict_one(xi) for xi in x])

# carat → price / 1-feature
xc = df["carat"].values; yc = y
tree = SimpleTree(max_depth=3).fit(xc, yc)
x_plot = np.linspace(xc.min(), xc.max(), 300)

from sklearn.tree import DecisionTreeRegressor
sk = DecisionTreeRegressor(max_depth=3).fit(xc.reshape(-1,1), yc)
print(f"从零树第一刀: carat < {tree.thresh:.3f}")
print(f"sklearn 树第一刀阈值: {sk.tree_.threshold[0]:.3f} → 一致 (都贪心选 MSE 最小切分)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xc, yc, alpha=0.1, s=6)
ax.plot(x_plot, tree.predict(x_plot), "r-", lw=2, label="从零树 (depth=3)")
ax.set_xlabel("carat"); ax.set_ylabel("price"); ax.legend()
ax.set_title("回归树: 阶梯状 (8 个叶子区域, 每段=区域均值)")
plt.tight_layout(); plt.show()
print("阶梯状预测 = 分段常数; depth=3 → 最多 2³=8 个叶子区域")


<a id="4"></a>
## 4. 对照 sklearn + 可视化切分 / sklearn & Visualizing Splits


In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# 浅树可视化 / visualize a shallow tree
tree2d = DecisionTreeRegressor(max_depth=2).fit(df[["carat","depth"]], y)
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(tree2d, feature_names=["carat","depth"], filled=True, rounded=True,
          fontsize=9, precision=0, ax=ax)
ax.set_title("决策树结构 (depth=2): 每个节点显示切分条件 + 区域均值")
plt.tight_layout(); plt.show()
print("读树: 根节点 carat<阈值? → 左右分支 → 叶子 value=该区域平均价")
print("squared_error 显示该节点的 MSE, samples 显示落入该节点的样本数")


<a id="5"></a>
## 5. 深度 = 偏差-方差; 剪枝 ⭐ / Depth & Pruning

**树极易过拟合**——深度不限时, 树会一直切到每个叶子只剩 1 个点（完美记住训练集, 灾难性方差）。控制手段：

| 超参 | 作用 |
|---|---|
| `max_depth` | 最大深度（最直接）|
| `min_samples_leaf` | 叶子最少样本数（防止切出极小叶子）|
| `min_samples_split` | 内部节点最少样本数才允许切 |
| `ccp_alpha` | **成本复杂度剪枝**：先长满再剪掉收益小的分支 |

**预剪枝**（提前停止, max_depth 等）vs **后剪枝**（ccp_alpha, 长满再剪）。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d in zip(axes, [2, 5, None]):
    t = DecisionTreeRegressor(max_depth=d, random_state=0).fit(xc.reshape(-1,1), yc)
    ax.scatter(xc, yc, alpha=0.08, s=6)
    ax.plot(x_plot, t.predict(x_plot.reshape(-1,1)), "r-", lw=1.5)
    tag = "欠拟合" if d==2 else ("刚好" if d==5 else "过拟合(每点一叶)")
    ax.set_title(f"max_depth={d}  {tag}")
plt.tight_layout(); plt.show()

# 深度 vs CV / depth selection
depths = [1,2,3,5,8,12,20,None]
cv = [cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=0), X, y, cv=5, scoring="r2").mean() for d in depths]
print("深度 vs CV R²:")
for d, s in zip(depths, cv): print(f"  max_depth={str(d):<6} CV R²={s:.3f}")
print("深度太浅欠拟合, 不限深度过拟合(CV 掉) → 选中等深度")


<a id="6"></a>
## 6. 特征重要性 + 不需缩放 / Importance & No Scaling

**特征重要性**: 每个特征在所有切分中带来的总方差下降, 归一化。
**不需缩放**: 验证缩放前后结果完全一样（3.4 节的承诺）。


In [ ]:
from sklearn.preprocessing import StandardScaler

tree = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X, y)
imp = pd.Series(tree.feature_importances_, index=feat_names).sort_values(ascending=False)
print("特征重要性 (方差下降贡献):")
print(imp.round(3))
print(f"\ncarat 主导 — 钻石价格主要由克拉决定, 符合常识\n")

# 验证不需缩放 / verify scaling invariance
raw_r2 = cross_val_score(DecisionTreeRegressor(max_depth=6, random_state=0), X, y, cv=5).mean()
Xs = StandardScaler().fit_transform(X)
scaled_r2 = cross_val_score(DecisionTreeRegressor(max_depth=6, random_state=0), Xs, y, cv=5).mean()
print(f"不缩放 CV R² = {raw_r2:.4f}")
print(f"缩放后 CV R² = {scaled_r2:.4f}")
print("完全相同 → 树对单调变换免疫, 不需缩放 (3.4 铁律的验证)")


<a id="7"></a>
## 7. 树的优缺点 → 为何集成 / Pros, Cons, Why Ensemble

| ✅ 优点 | ❌ 缺点 |
|---|---|
| 不需缩放/可处理混合类型 | **高方差**(数据微变→树结构大变) ⭐ |
| 天然抓交互/非线性 | **易过拟合** |
| 可解释(if-else 规则) | 单棵树**预测精度有限** |
| 抗维度诅咒 | 阶梯预测**不平滑**(无法外推) |
| 快 | 贪心切分**非全局最优** |

**核心矛盾**: 单棵树**高方差**——同样的数据稍变, 切分点全变, 树结构面目全非。

**解决方案 = 集成(ensemble)**:
- **随机森林(4.12)**: 训练**很多棵树取平均** → 方差骤降（bagging, 0.10 偏差方差的方差侧）
- **GBDT(4.13)**: 训练很多棵树**依次纠错** → 偏差骤降（boosting）

单棵树是弱学习器, 但**集成起来是表格数据之王**。这就是为什么 4.11 是 4.12/4.13 的必经之路。
A single tree is high-variance; ensembles (forest = average many, GBDT = correct sequentially) make trees the kings of tabular data.


<a id="8"></a>
## 8. 小结 / Summary

```
回归树 = 递归切分空间 + 叶子取区域均值 → 阶梯状预测
切分准则: 贪心选 (特征,阈值) 使加权 MSE 最小 (=方差下降最多), CART
优点: 不需缩放(单调变换免疫) + 抓交互 + 抗维度诅咒 + 可解释
控过拟合: max_depth / min_samples_leaf / ccp_alpha(后剪枝)
特征重要性 = 方差下降总贡献
致命缺点: 高方差(数据微变树大变) → 集成救场:
  随机森林(4.12, 平均降方差) / GBDT(4.13, 纠错降偏差)
```

### 💡 面试速查
1. **回归树切分**: 贪心最小化加权 MSE (=方差); 分类树用基尼/熵
2. **树不需缩放**: 切分看阈值, 单调变换不变
3. **树高方差易过拟合** → max_depth/剪枝控制 → 集成根治
4. **特征重要性** = 方差下降贡献
5. **单棵树是弱学习器**; 集成才是表格之王

### 下一节
**4.12 随机森林回归**——把很多棵高方差的树**平均**起来, 方差骤降。bagging + 特征随机 + OOB 评估, 几乎零调参的强力模型。
